# Imports & env load

In [43]:
import chromadb
from langchain_chroma import Chroma
from groq import Groq
from dotenv import load_dotenv
import os 

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
client = Groq()

## Embedding Model

In [44]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

### Model Selection

In [45]:
model_name = 'openai/gpt-oss-120b'

# Query Expansion

## Vector Database Connection

In [46]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [47]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [48]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [49]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

## Vector database retrival

In [50]:
retriever = vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={
        'k': 3
    }
)

## Prompt for Query expansion

In [51]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [52]:
benchmark_questions = [
    {
        "question_id": "Q1",
        "question": "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A."
    },
    {
        "question_id": "Q2",
        "question": "Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims."
    },
    {
        "question_id": "Q3",
        "question": "A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations."
    },
    {
        "question_id": "Q4",
        "question": "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"
    }
]

In [ ]:
# user_query = "What was the automotive revenue in 2021?"

## Final Query Expansion Result

In [53]:
final_query_system = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [54]:
final_query_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [57]:
def run_pipeline(question_id, user_query):

    # --------------------------------------------------
    # 1. Query Expansion (INSIDE FUNCTION)
    # --------------------------------------------------
    prompt = [
        {"role": "system", "content": query_expansion_system_message},
        {"role": "user", "content": user_message_template.format(question=user_query)}
    ]

    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0.2
    )

    query_expansion_list = [
        q.strip()
        for q in response.choices[0].message.content.split("\n")
        if q.strip()
    ]

    # --------------------------------------------------
    # 2. Baseline Retrieval
    # --------------------------------------------------
    baseline_docs = retriever.invoke(user_query)

    baseline_top_chunks = []
    for rank, doc in enumerate(baseline_docs):
        baseline_top_chunks.append({
            "source": doc.metadata.get("source"),
            "page": doc.metadata.get("page"),
            "page_label": doc.metadata.get("page_label"),
            "rank": rank + 1,
            "content_preview": doc.page_content[:150]
        })

    # --------------------------------------------------
    # 3. Expanded Retrieval
    # --------------------------------------------------
    expanded_context_list = []
    expanded_top_chunks = []
    seen_chunks = {}

    for query in query_expansion_list:

        docs = retriever.invoke(query)

        for rank, doc in enumerate(docs):

            chunk_text = doc.page_content

            clean_text = (
                chunk_text
                .replace("\n", " ")
                .replace("\t", " ")
                .replace("\r", " ")
            )

            clean_text = " ".join(clean_text.split())

            if chunk_text not in seen_chunks:

                expanded_context_list.append(chunk_text)

                seen_chunks[chunk_text] = {
                    "source": doc.metadata.get("source"),
                    "page": doc.metadata.get("page"),
                    "page_label": doc.metadata.get("page_label"),
                    "retrieved_by": [query],
                    "rank": rank + 1,
                    "content_preview": clean_text[:200]
                }

            else:
                seen_chunks[chunk_text]["retrieved_by"].append(query)

    expanded_top_chunks = list(seen_chunks.values())

    # --------------------------------------------------
    # 4. Final Answer Generation
    # --------------------------------------------------
    final_context = "\n\n---\n\n".join(map(str, expanded_context_list))

    prompt = [
        {"role": "system", "content": final_query_system},
        {"role": "user", "content": final_query_user_message_template.format(
            context=final_context,
            question=user_query
        )}
    ]

    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()

    # --------------------------------------------------
    # 5. Cleaning
    # --------------------------------------------------
    def clean_text(text):
        if not isinstance(text, str):
            return text
        text = text.replace("\n", " ")
        text = text.replace("\t", " ")
        text = text.replace("\r", " ")
        text = text.replace("\\", "/")
        return " ".join(text.split())

    # clean baseline + expanded
    for chunk in baseline_top_chunks:
        chunk["source"] = clean_text(chunk.get("source"))

    for chunk in expanded_top_chunks:
        chunk["source"] = clean_text(chunk.get("source"))

    # --------------------------------------------------
    # 6. Citations
    # --------------------------------------------------
    citations = []
    seen = set()

    for chunk in expanded_top_chunks:

        key = (chunk.get("source"), chunk.get("page"))

        if key not in seen:
            citations.append({
                "source": chunk.get("source"),
                "page": chunk.get("page"),
                "page_label": chunk.get("page_label")
            })
            seen.add(key)

    # --------------------------------------------------
    # 7. Final Result
    # --------------------------------------------------
    result = {
        "question_id": question_id,
        "original_query": user_query,
        "expanded_queries": query_expansion_list,
        "baseline_top_chunks": baseline_top_chunks,
        "expanded_top_chunks": expanded_top_chunks,
        "final_answer": prediction,
        "citations": citations,
        "retrieval_improvement_analysis":
            f"Expanded retrieval used {len(query_expansion_list)} query variations."
    }

    return result

In [58]:
results = []

for item in benchmark_questions:

    print("Running:", item["question_id"])

    output = run_pipeline(
        item["question_id"],
        item["question"]
    )

    results.append(output)

Running: Q1
Running: Q2
Running: Q3
Running: Q4


In [59]:
import json

with open("query_expansion_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Saved benchmark_results.json")

Saved benchmark_results.json
